# Plan-Execute-Replan 에이전트


이번 시간에 볼 패턴:

- `planner`: 사용자 목표를 실행 가능한 단계 목록으로 변환
- `execute`: 현재 단계 하나를 도구 사용 에이전트로 실행
- `replan`: 완료한 단계와 남은 계획을 보고 다음 행동 결정
- `final_report`: 실행 기록을 모아 최종 보고서 생성


## 환경 준비



In [ ]:
from dotenv import load_dotenv

load_dotenv()


## 1. 검색 도구와 실행 에이전트

- `planner`는 계획만 만들고, `agent_executor`는 계획 중 현재 단계 하나를 실제로 실행합니다. 여기서는 Tavily 검색 도구를 연결해 리서치형 질문에 답할 수 있게 구성합니다.


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

search_tool = 

tools = 

agent_executor = 



## 2. 전체 흐름

- Plan-Execute-Replan 구조는 한 번에 답을 만들기보다, 먼저 계획을 세우고 실행 결과를 바탕으로 계획을 계속 줄여나갑니다. 충분한 근거가 쌓이면 최종 보고서 노드로 이동합니다.


<div style="max-width: 900px; margin: 12px 0;">
<svg viewBox="0 0 900 250" width="100%" role="img" aria-label="Plan execute replan flow diagram" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="arrow-plan-execute" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#495057" />
    </marker>
    <style>
      .node { fill:#eef6ff; stroke:#2f80ed; stroke-width:1.5; rx:8; }
      .agent { fill:#fff3cd; stroke:#f08c00; stroke-width:1.8; rx:8; }
      .decision { fill:#f8f9fa; stroke:#868e96; stroke-width:1.5; }
      .done { fill:#e6fcf5; stroke:#12b886; stroke-width:1.5; rx:8; }
      .txt { font: 14px sans-serif; fill:#212529; text-anchor:middle; dominant-baseline:middle; }
      .small { font: 12px sans-serif; fill:#495057; text-anchor:middle; dominant-baseline:middle; }
      .edge { stroke:#495057; stroke-width:1.6; fill:none; marker-end:url(#arrow-plan-execute); }
    </style>
  </defs>
  <rect class="node" x="25" y="103" width="80" height="44"/><text class="txt" x="65" y="125">START</text>
  <rect class="node" x="145" y="88" width="115" height="74"/><text class="txt" x="202" y="113">planner</text><text class="small" x="202" y="137">계획 생성</text>
  <rect class="agent" x="305" y="88" width="120" height="74"/><text class="txt" x="365" y="113">execute</text><text class="small" x="365" y="137">현재 단계 실행</text>
  <rect class="node" x="470" y="88" width="115" height="74"/><text class="txt" x="527" y="113">replan</text><text class="small" x="527" y="137">계획 갱신</text>
  <polygon class="decision" points="670,76 735,125 670,174 605,125"/><text class="txt" x="670" y="118">완료?</text><text class="small" x="670" y="140">response</text>
  <rect class="done" x="780" y="88" width="95" height="74"/><text class="txt" x="827" y="113">final</text><text class="small" x="827" y="137">보고서 생성</text>
  <path class="edge" d="M105 125 H145"/>
  <path class="edge" d="M260 125 H305"/>
  <path class="edge" d="M425 125 H470"/>
  <path class="edge" d="M585 125 H605"/>
  <path class="edge" d="M735 125 H780"/><text class="small" x="756" y="113">yes</text>
  <path class="edge" d="M670 174 C650 220 360 220 365 162"/><text class="small" x="515" y="210">no, 다음 단계 실행</text>
</svg>
</div>


## 3. 상태와 구조화 출력 모델

- `passed_steps`는 여러 실행 결과가 누적되는 필드라서 `operator.add` reducer를 붙입니다. `Plan`, `Response`, `Act`는 LLM이 반환해야 할 JSON 형태를 명확하게 제한합니다.


In [ ]:
import operator
from typing import Annotated, TypedDict, Union

from pydantic import BaseModel, Field


class PlanExecuteState(TypedDict):
    input: Annotated[str, "사용자 요청"]
    plan: Annotated[list[str], "현재 남아 있는 계획"]
    passed_steps: Annotated[list[tuple[str, str]], operator.add]
    response: Annotated[str, "최종 응답"]


class Plan(BaseModel):
    """ """

    steps: list[str] = Field(
        description="목표를 달성하기 위해 순서대로 실행해야 하는 단계 목록"
    )


class Response(BaseModel):
    """ """

    response: str


class Act(BaseModel):
    """ """

    action: Union[Response, Plan] = Field(
        description=(
            "사용자에게 바로 답할 수 있으면 Response를 사용하고, "
            "추가 실행이 필요하면 Plan을 사용합니다."
        )
    )


## 4. 계획 수립 Planner

- 첫 번째 LLM 호출은 답변을 만들지 않고 계획만 만듭니다. 각 단계가 독립적으로 실행 가능해야 이후 `execute` 노드가 안정적으로 한 단계씩 처리할 수 있습니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """주어진 목표를 달성하기 위한 간단한 단계별 계획을 세우세요.
이 계획은 각 단계를 올바르게 실행하면 정확한 답을 얻을 수 있는 개별 작업들로 구성되어야 합니다.
불필요한 단계는 추가하지 마세요.
마지막 단계의 결과가 최종 답변이 되어야 합니다.
각 단계에 필요한 정보가 모두 포함되도록 하고, 단계를 건너뛰지 마세요.
한국어로 답변하세요.""",
        ),
        ("placeholder", "{messages}"),
    ]
)

planner = 

In [ ]:
sample_plan = planner.invoke(
    {
        "messages": [
            (
                "user",
                " ",
            )
        ]
    }
)



## 5. 재계획 Replanner

재계획 노드는 지금까지 완료한 작업을 보고 두 가지 중 하나를 선택합니다.

- 아직 해야 할 일이 남았으면 `Plan(steps=[...])`
- 충분히 답할 수 있으면 `Response(response=...)`


In [ ]:
replanner_prompt = ChatPromptTemplate.from_template(
    """주어진 목표에 맞게 남은 계획을 갱신하세요.
원래 계획은 사용자의 목표에 답하기 위해 설계되었습니다.
아직 반드시 수행해야 하는 단계만 포함하세요.
이미 완료한 단계는 다시 반환하지 마세요.

사용자 목표:
{input}

현재 남은 계획:
{plan}

완료한 단계와 결과:
{passed_steps}

더 이상 필요한 단계가 없고 사용자에게 답변할 수 있다면 Response로 응답하세요.
그렇지 않다면 남은 단계만 포함한 Plan을 반환하세요.
한국어로 답변하세요."""
)

replanner = 


## 6. 노드 함수 만들기

- `plan_step`은 최초 계획을 만들고, 
- `execute_step`은 맨 앞의 계획 하나만 실행합니다. 
- `replan_step`은 남은 계획을 갱신하거나 최종 응답 준비 상태로 바꿉니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser


def format_plan(plan: list[str]) -> str:
    return "\n".join(f"{idx}. {step}" for idx, step in enumerate(plan, start=1))


def format_passed_steps(passed_steps: list[tuple[str, str]]) -> str:
    if not passed_steps:
        return "아직 완료한 단계가 없습니다."
    return "\n\n".join(
        f"Step {idx}: {task}\nResult: {result}"
        for idx, (task, result) in enumerate(passed_steps, start=1)
    )


def plan_step(state: PlanExecuteState) -> dict:
    plan = planner.invoke({"messages": [("user", state["input"])]})
    return {"plan": plan.steps}


def execute_step(state: PlanExecuteState) -> dict:
    plan = state["plan"]
    task = plan[0]
    plan_text = format_plan(plan)
    task_formatted = f"""다음 계획을 기준으로 작업하세요:
{plan_text}

당신이 실행해야 할 작업은 1단계입니다: {task}

결과를 한국어로 반환하세요."""

    agent_response = agent_executor.invoke({"messages": [("user", task_formatted)]})
    result = agent_response["messages"][-1].content
    return {"passed_steps": [(task, result)]}


def replan_step(state: PlanExecuteState) -> dict:
    output = replanner.invoke(
        {
            "input": state["input"],
            "plan": format_plan(state["plan"]),
            "passed_steps": format_passed_steps(state["passed_steps"]),
        }
    )

    if isinstance(output.action, Response):
        return {"response": output.action.response}

    if not output.action.steps:
        return {"response": "더 실행할 단계가 없습니다."}

    return {"plan": output.action.steps}


def should_end(state: PlanExecuteState) -> str:
    if state.get("response"):
        return "final_report"
    return "execute"


## 7. 최종 보고서 노드

- 재계획 단계에서 충분히 답할 수 있다고 판단되면, 지금까지 실행한 단계와 결과를 하나의 마크다운 보고서로 정리합니다.


In [ ]:
final_report_prompt = ChatPromptTemplate.from_template(
    """목표와 이전에 완료한 단계가 주어집니다.
마크다운 형식의 최종 보고서를 작성하세요.
전문적인 어조를 사용하고 한국어로 작성하세요.

목표:
{input}

완료한 단계:
{passed_steps}

최종 보고서:"""
)

final_report =


def generate_final_report(state: PlanExecuteState) -> dict:
    response = final_report.invoke(
        {
            "input": state["input"],
            "passed_steps": format_passed_steps(state["passed_steps"]),
        }
    )
    return {"response": response}


## 8. 그래프 생성

- 그래프는 `planner -> execute -> replan` 순서로 진행됩니다. 
- `replan` 이후에는 조건부 엣지로 다음 실행을 반복하거나 최종 보고서로 이동합니다.


In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


def show_graph(app, *, xray: bool = False) -> None:
    graph_view = app.get_graph(xray=xray)
    try:
        display(Image(graph_view.draw_mermaid_png()))
    except Exception as exc:
        print(f"그래프 이미지 생성 실패: {exc}")
        print(graph_view.draw_mermaid())


In [ ]:

graph_builder = 

graph_builder.add_node("planner", )
graph_builder.add_node("execute", )
graph_builder.add_node("replan", )
graph_builder.add_node("final_report", )

graph_builder.add_edge()
graph_builder.add_edge()
graph_builder.add_edge()
graph_builder.add_conditional_edges(
    
)
graph_builder.add_edge()

graph = graph_builder.compile()


In [ ]:
show_graph(graph, xray=True)


## 9. 실행 예시

- `stream_mode="values"`로 실행하면 각 노드가 상태를 어떻게 바꾸는지 단계별로 확인할 수 있습니다. 실제 검색과 LLM 호출이 발생하므로 API 키와 네트워크 연결이 필요합니다.


In [ ]:
from pprint import pprint

config = {
    "recursion_limit": 50,
    "configurable": {"thread_id": "plan-execute-demo-001"},
}

inputs = {
    "input": "2026년에 주목할 만한 AI 관련 주식 후보를 조사하고 근거를 정리해줘",
    "plan": [],
    "passed_steps": [],
    "response": "",
}

events = graph.stream(inputs, config=config, stream_mode="values")

for event in events:
    print("=" * 80)
    pprint(event)


## 10. 정리
- 실무에서도 이런 식으로 많이 씁니다. LangChain으로 LLM 호출, 도구 호출, 파서, 프롬프트 체인을 만들고, LangGraph로 반복, 분기, 상태 저장, 재시도, human-in-the-loop 같은 흐름을 제어합니다.
- Plan-Execute-Replan은 복잡한 요청을 작은 단계로 나누고, 실행 결과를 보며 계획을 갱신하는 패턴입니다.
- `planner`는 전체 계획을 만들고, `execute`는 한 번에 하나의 단계만 수행합니다.
- `passed_steps`에는 완료한 작업과 결과가 누적되므로 reducer가 필요합니다.
- `replan`은 남은 계획을 줄이거나 최종 응답으로 넘어갈지 결정합니다.
- 최종 보고서 노드를 분리하면 중간 실행 결과를 더 일관된 형식으로 정리할 수 있습니다.


## [실습]

1. `inputs["input"]`을 투자 질문이 아닌 여행 계획, 시장 조사, 기술 비교 질문으로 바꿔 실행해보세요.
2. `TavilySearch(max_results=3)`의 결과 개수를 5로 늘리고 최종 보고서가 어떻게 달라지는지 확인하세요.
3. `planner_prompt`에 "최대 3단계로 계획하라"는 조건을 추가해 반복 횟수를 줄여보세요.
4. `execute_step`에서 `task_formatted` 문구를 바꿔 실행 노드의 답변 품질이 어떻게 바뀌는지 비교하세요.
5. `should_end` 조건을 수정해서 `passed_steps`가 3개 이상이면 강제로 최종 보고서로 이동하게 만들어보세요.
6. `generate_final_report`가 출처 목록, 리스크, 결론을 별도 섹션으로 반드시 포함하도록 프롬프트를 강화해보세요.
